# OP-05 · Températures C3S par période

**Notebook opérationnel** — à exécuter chaque mois, après `OP_01`.

| | |
|---|---|
| Étape du workflow | E2 → entrée de E4 et E5 |
| Entrées | fichiers bruts C3S (`TMAX`, `TMIN` quotidiens ; `TEMP` mensuel) |
| Sorties | `DATA_OSF/derived/c3s/YYYYMM/c3s_<centre>_<t2m\|tmax\|tmin>_<forecast\|hindcast>_periods.nc` |
| Durée | environ 10 min pour 7 modèles |

- **Tmax et Tmin** viennent du jeu quotidien : décades, mois et saisons.
- **T2m moyenne** vient des statistiques mensuelles : mois et saisons seulement. Ce jeu fournit l'ensemble décalé complet d'UKMO et de BoM, d'où un nombre de membres plus élevé pour ces deux modèles.

Équivalent en ligne de commande :
```
python scripts/run_download_c3s.py --config config/cycle_YYYYMM.yaml --variable tmax
python scripts/run_download_c3s.py --config config/cycle_YYYYMM.yaml --variable tmin
python scripts/run_download_c3s.py --config config/cycle_YYYYMM.yaml --variable t2m
python scripts/run_c3s_temperature.py --config config/cycle_YYYYMM.yaml
```

## Paramètres

In [ ]:
CYCLE_CONFIG = "config/cycle_202609.yaml"
VARIABLES    = ["t2m", "tmax", "tmin"]
MODELS       = None      # None = tous les modèles de la configuration
TELECHARGER  = True      # False : ne traiter que les fichiers déjà présents

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
from eccas_s2s.settings import load_cycle
cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")

## 1. Téléchargement (les fichiers déjà présents sont ignorés)

In [ ]:
from eccas_s2s.operations import download_c3s
downloads = []
if TELECHARGER:
    for v in VARIABLES:
        downloads.append(download_c3s.run(CYCLE_CONFIG, v, models=MODELS))
        print(f"{v} : {len(downloads[-1].outputs)} fichier(s), "
              f"{len(downloads[-1].parameters.get('failures', {}))} échec(s)")

## 2. Valeurs par période

In [ ]:
from eccas_s2s.operations import c3s_temperature
ctx = c3s_temperature.run(CYCLE_CONFIG, variables=VARIABLES, models=MODELS)
print(f"\nStatut : {ctx.status}")
for w in ctx.warnings:
    print(" ⚠", w)

## 3. Récapitulatif

In [ ]:
from eccas_s2s.operations.c3s_totals import derived_dir
summary = pd.read_csv(derived_dir(cfg) / "c3s_temperature_summary.csv")
display(summary[summary.variable.isin(VARIABLES)])
print("couples (année, période) sans valeur :", int(summary.empty_year_periods.sum()))

## 4. Contrôle visuel : Tmax de la première saison

In [ ]:
from eccas_s2s.viz.maps import map_panel
from eccas_s2s.operations.c3s_totals import load_totals
fields, titles = [], []
for c in cfg.c3s_models:
    try:
        da = load_totals(cfg, c, "forecast", "tmax").sel(period="season_m0").isel(year=0)
    except (FileNotFoundError, KeyError):
        continue
    fields.append(da.mean("number")); titles.append(cfg.c3s_models[c].label)
if fields:
    fig = map_panel(fields, titles, shapefile=cfg.raw["paths"]["shapefile"], cmap="YlOrRd",
                    levels=[18, 22, 26, 28, 30, 32, 34, 36, 38, 40], extend="both", ncols=4,
                    cbar_label="°C", suptitle="Tmax moyen de la première saison — prévision brute")

In [ ]:
for c in downloads + [ctx]:
    print(f"{c.step:24s} {c.status:8s} {c.run_dir / 'manifest.json'}")